## BRT
1. Build presences/absences
2. Fit one full model + vessel-type-specific models
3. Extract relative importance of each driver, partial dependence plots
5. Tune hyperparameters

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from sklearn.inspection import PartialDependenceDisplay

from brt_modeling import (
    build_presence_absence, debug_presence_absence, sample_rasters,
    get_raster_paths, rebalance_1to1, run_model,
)

gear_types = ["TRAWLERS", "DRIFTING_LONGLINES", "SQUID_JIGGER", "TUNA_PURSE_SEINES", None] # None = all gear types
period = "2017_2019"
gear_type = gear_types[4]
df_gaps = pd.read_csv(f"data/processed/gaps_{period}_filtered.csv")
df_fishing = pd.read_csv(f"data/raw/fishing_gridded_{period}.csv")
df_fishing

In [ ]:
df_gaps

## Build presences/ absences

In [ ]:
# pa_df = build_presence_absence(df_gaps, df_fishing)
# debug_presence_absence(pa_df)

## Extract drivers at presences/ absences grid cells

In [ ]:
raster_paths = get_raster_paths(period)
feature_cols = list(raster_paths.keys())

# model_df = sample_rasters(pa_df, raster_paths)
# # loitering_hours: NaN = no loitering recorded = 0 hours, not missing data
# model_df['loitering_hours'] = model_df['loitering_hours'].fillna(0)
# model_df.to_csv(f"data/processed/model_data_{period}.csv", index=False)

# # check for nan
# nan_summary = model_df[list(raster_paths.keys())].isna().sum()
# print(nan_summary)

# nan_rows = model_df[model_df[list(raster_paths.keys())].isna().any(axis=1)]
# print(f"{len(nan_rows)} / {len(model_df)} rows affected")
# print(nan_rows.groupby('presence').size())  # check class balance impact

# model_df = rebalance_1to1(model_df)
# print(model_df.groupby('presence').size())

# Run model per geartype and save results

In [ ]:
model_df, model, metrics = run_model(df_gaps, df_fishing, feature_cols, raster_paths, gear_type=gear_type)

In [ ]:
PartialDependenceDisplay.from_estimator(model, model_df[feature_cols], ["loitering_hours"])